# Data Rescue Kit
## MAD 2502 — Applied Data

This is not meant to be read straight through like a lecture. It is a **problem-solving notebook** for the moments when your capstone data do something annoying.

Use the section that matches your problem.


In [ ]:
from io import StringIO
from pathlib import Path
import json

import pandas as pd


## Problem 1 — Python cannot find my file


In [ ]:
print("Current working folder:")
print(Path.cwd())

print("\nFiles here:")
for path in list(Path.cwd().iterdir())[:10]:
    print(" -", path.name)


Use `Path.cwd()` to see where Python is looking. Prefer paths relative to your project folder instead of hard-coded paths tied to one computer.

```python
data_path = Path("data") / "my_file.csv"
```


## Problem 2 — My numbers are stored as text


In [ ]:
messy = pd.DataFrame({
    "price": ["$1,250", "$800", "unknown", "$2,100"],
    "rating": ["4.7", "4.1", "", "4.9"],
})

messy


In [ ]:
messy["price_clean"] = (
    messy["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

messy["price_clean"] = pd.to_numeric(
    messy["price_clean"],
    errors="coerce",
)

messy["rating_clean"] = pd.to_numeric(
    messy["rating"],
    errors="coerce",
)

messy


`errors="coerce"` turns values that cannot be interpreted as numbers into `NaN`. That is often safer than silently inventing a number.


## Problem 3 — I have missing values


In [ ]:
print(messy.isna().sum())

print("\nRows with a missing cleaned price:")
display(messy[messy["price_clean"].isna()])


In [ ]:
# Examples of different choices:
drop_missing = messy.dropna(subset=["price_clean"])
filled_rating = messy["rating_clean"].fillna(
    messy["rating_clean"].median()
)

print(drop_missing)
print(filled_rating)


There is no universal “correct” missing-data command. Decide what missing means **before** choosing `dropna` or `fillna`.


## Problem 4 — My column names are terrible


In [ ]:
df = pd.DataFrame({
    " Event Name ": ["A", "B"],
    "Total Cost ($)": [10, 20],
    "Number of People": [50, 80],
})

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("($)", "", regex=False)
)

df


## Problem 5 — I have duplicate rows


In [ ]:
duplicates = pd.DataFrame({
    "name": ["alpha", "beta", "beta", "gamma"],
    "value": [10, 20, 20, 30],
})

print(duplicates.duplicated())
display(duplicates.drop_duplicates())


## Problem 6 — My dates are just strings


In [ ]:
dates = pd.DataFrame({
    "date": ["2026-10-01", "2026-10-05", "bad date", "2026-10-21"],
    "value": [5, 7, 8, 12],
})

dates["date"] = pd.to_datetime(
    dates["date"],
    errors="coerce",
)

dates


In [ ]:
dates["month"] = dates["date"].dt.month
dates["day_of_week"] = dates["date"].dt.day_name()
dates


## Problem 7 — I need to combine several files


In [ ]:
first = pd.DataFrame({"name": ["A", "B"], "score": [80, 90]})
second = pd.DataFrame({"name": ["C", "D"], "score": [85, 95]})

combined = pd.concat(
    [first, second],
    ignore_index=True,
)

combined


## Problem 8 — My JSON is nested


In [ ]:
nested = {
    "project": "example",
    "records": [
        {"name": "A", "stats": {"x": 3, "y": 8}},
        {"name": "B", "stats": {"x": 5, "y": 13}},
    ],
}

flat = pd.json_normalize(nested["records"])
flat


## Problem 9 — My DataFrame is empty


In [ ]:
example = pd.DataFrame({"value": [1, 2, 3]})

filtered = example[example["value"] > 100]

print("shape:", filtered.shape)
print("empty:", filtered.empty)
display(filtered)


When a DataFrame is unexpectedly empty, check the pipeline one piece at a time:

1. Did the request/file load return anything?
2. What is `df.shape` before filtering?
3. What unique values are actually in the column?
4. Is the column numeric or text?
5. Is your condition too restrictive?


## Problem 10 — I don't know what object I have


In [ ]:
sample = pd.DataFrame({
    "category": ["A", "B", "A"],
    "value": [1, 2, 3],
})

print(type(sample))
print(type(sample["value"]))
print(sample.dtypes)


# A good debugging routine

When something fails, do not rewrite everything at once.

```text
1. Read the error message.
2. Find the first line of YOUR code named in the traceback.
3. Print the object just before that line.
4. Check type(...), shape, columns, and a few values.
5. Fix one assumption.
6. Run again.
```

### Useful inspection commands

```python
type(x)
df.shape
df.columns
df.dtypes
df.head()
df.info()
df.isna().sum()
df["column"].unique()
df["column"].value_counts(dropna=False)
```

A large fraction of data debugging is discovering that the object is not shaped or typed the way you assumed.
